In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


In [2]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

CUDA available: True
Device: NVIDIA H200 NVL
Using device: cuda


# Consistency Evaluation for erasing-llm_eval

This notebook evaluates the consistency of the research project at `/net/scratch2/smallyan/erasing-llm_eval` against its stated goals and plan.

## Evaluation Criteria

### CS1. Conclusion vs Original Results
- **PASS** — All evaluable conclusions in the documentation match the results originally recorded in the code implementation notebook.
- **FAIL** — At least one evaluable conclusion contradicts the originally recorded results.

### CS2. Implementation Follows the Plan
- **PASS** — A Plan file exists and all plan steps appear in the implementation.
- **FAIL** — A Plan file exists and at least one plan step is missing in the implementation.

In [3]:
# First, let's explore the repository structure
repo_path = '/net/scratch2/smallyan/erasing-llm_eval'

# List all files and directories
for root, dirs, files in os.walk(repo_path):
    level = root.replace(repo_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        print(f'{subindent}{file}')

erasing-llm_eval/
  documentation.pdf
  .gitignore
  __init__.py
  CodeWalkthrough.md
  requirements.txt
  plan.md
  trainscripts/
    erase.py
    prepare_consistency_data.py
    __init__.py
  utils/
    metrics.py
    __init__.py
    lora.py
    __pycache__/
      lora.cpython-311.pyc
      __init__.cpython-311.pyc
      metrics.cpython-311.pyc
  data/
    wmdp-keywords.json
    harrypotter/
      hp-questions-dual.json
      hp-questions.json
      .ipynb_checkpoints/
        old-hp-questions-checkpoint.json
        hp-questions-checkpoint.json
        EASY_hp_trivia_1239-checkpoint.jsonl
    wmdp/
      bio-questions.json
      chem-questions.json
      cyber-questions.json
  notebooks/
    inference.ipynb
  .git/
    FETCH_HEAD
    ORIG_HEAD
    config
    description
    index
    HEAD
    COMMIT_EDITMSG
    packed-refs
    hooks/
      push-to-checkout.sample
      update.sample
      pre-merge-commit.sample
      pre-receive.sample
      prepare-commit-msg.sample
      pre-appl

## Repository Structure

The repository contains:
- `plan.md` - The plan file
- `documentation.pdf` - Project documentation
- `CodeWalkthrough.md` - Code walkthrough
- `notebooks/inference.ipynb` - Implementation notebook
- `trainscripts/` - Training scripts (erase.py, prepare_consistency_data.py)
- `utils/` - Utility modules (metrics.py, lora.py)
- `data/` - Data files for evaluation

In [4]:
# Read the plan file
plan_path = os.path.join(repo_path, 'plan.md')
with open(plan_path, 'r') as f:
    plan_content = f.read()
print("=" * 80)
print("PLAN.MD CONTENT")
print("=" * 80)
print(plan_content)

PLAN.MD CONTENT
# Plan
## Objective
To develop a principled approach for erasing broad conceptual knowledge from language models by leveraging the model's own introspective classification capabilities to reduce generation probabilities for concept-specific content while preserving broader model capabilities.

## Hypothesis
1. Language models can act as their own critics to evaluate whether text belongs to a particular concept, enabling self-classification as a natural objective for unlearning.
2. Effective concept erasure requires modifying the model to reduce the likelihood of generating text it would classify as containing the target concept, rather than reversing gradients or manipulating representations.
3. Low-rank adapters applied to early model layers enable precise knowledge modification while maintaining broader capabilities.

## Methodology
1. ELM uses introspective classification by leveraging implicit model probabilities with two context prompts: c− representing the concept

In [5]:
# Read the CodeWalkthrough.md file
walkthrough_path = os.path.join(repo_path, 'CodeWalkthrough.md')
with open(walkthrough_path, 'r') as f:
    walkthrough_content = f.read()
print("=" * 80)
print("CODE WALKTHROUGH CONTENT")
print("=" * 80)
print(walkthrough_content)

CODE WALKTHROUGH CONTENT
# Erasing Conceptual Knowledge from Language Models
###  [Project Website](https://elm.baulab.info) | [Arxiv Preprint](https://arxiv.org/pdf/2410.02760) | [Trained Models](https://elm.baulab.info/models/elm-wmdp/) | [Huggingface Models](https://huggingface.co/collections/baulab/elm-6715d68576da0cd1a89c0c04)<br>

<div align='center'>
<img src = 'images/method.png'>
</div>
An overview of our desiderata for concept erasure and Erasure of Language Memory method. The erased model must stay innocent of the erased concept, while still being fluent when prompted for the concept indicating seamless edit. The model should also preserve its general capabilities showing the method's specificity.

## Use Pretrained Models on Huggingface
We released our models on huggingface [here](https://huggingface.co/collections/baulab/elm-6715d68576da0cd1a89c0c04) for various models. To use one of the models: 
```
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [6]:
# Read the inference notebook
import json

notebook_path = os.path.join(repo_path, 'notebooks', 'inference.ipynb')
with open(notebook_path, 'r') as f:
    notebook_content = json.load(f)

print("=" * 80)
print("INFERENCE NOTEBOOK CONTENT")
print("=" * 80)

# Extract and display the content of each cell
for i, cell in enumerate(notebook_content['cells']):
    print(f"\n--- Cell {i+1} ({cell['cell_type']}) ---")
    source = ''.join(cell['source'])
    print(source[:2000] if len(source) > 2000 else source)
    
    # If there are outputs, show them
    if 'outputs' in cell and cell['outputs']:
        print("\n>>> OUTPUTS:")
        for output in cell['outputs']:
            if 'text' in output:
                text = ''.join(output['text'])
                print(text[:1500] if len(text) > 1500 else text)
            elif 'data' in output and 'text/plain' in output['data']:
                text = ''.join(output['data']['text/plain'])
                print(text[:1500] if len(text) > 1500 else text)

INFERENCE NOTEBOOK CONTENT

--- Cell 1 (code) ---
import os
# os.environ['HF_HOME']='../../hf_cache'
# os.environ['TRANSFORMERS_CACHE']='../../hf_cache'
from transformers import AutoModelForCausalLM, AutoTokenizer
import datasets
from tqdm.notebook import tqdm
import numpy as np
import torch
# from transformers import AdamW
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss,MSELoss, NLLLoss, KLDivLoss
import json
import random
import matplotlib.pyplot as plt
import transformers
import sys
sys.path.append('../.')
from utils.lora import LoRANetwork
from utils.metrics import get_wmdp_accuracy, get_mmlu_accuracy, get_truthfulqa, get_hp_accuracy
from peft import PeftModel, PeftConfig
transformers.utils.logging.set_verbosity(transformers.logging.CRITICAL)

--- Cell 2 (code) ---
# model_id = 'meta-llama/Meta-Llama-3-8B-Instruct'
# model_id = 'meta-llama/Llama-2-7b-hf'
# model_id = 'meta-llama/Llama-2-7b-chat-hf'
# model_id = 'mistralai/Mistral-7B-v0.1'
# model_id = 'EleutherA

In [7]:
# Continue reading the notebook cells
for i, cell in enumerate(notebook_content['cells'][5:], start=6):
    print(f"\n--- Cell {i} ({cell['cell_type']}) ---")
    source = ''.join(cell['source'])
    print(source[:2000] if len(source) > 2000 else source)
    
    # If there are outputs, show them
    if 'outputs' in cell and cell['outputs']:
        print("\n>>> OUTPUTS:")
        for output in cell['outputs']:
            if 'text' in output:
                text = ''.join(output['text'])
                print(text[:2000] if len(text) > 2000 else text)
            elif 'data' in output and 'text/plain' in output['data']:
                text = ''.join(output['data']['text/plain'])
                print(text[:2000] if len(text) > 2000 else text)

In [8]:
# Read the erase.py training script
erase_script_path = os.path.join(repo_path, 'trainscripts', 'erase.py')
with open(erase_script_path, 'r') as f:
    erase_content = f.read()
print("=" * 80)
print("ERASE.PY CONTENT")
print("=" * 80)
print(erase_content)

ERASE.PY CONTENT
import os
# os.environ['HF_HOME']='../../hf_cache'
# os.environ['TRANSFORMERS_CACHE']='../../hf_cache'
# os.environ['WANDB_DATA_DIR']='../../wandb_cache'
# os.environ['WANDB_API_KEY']='<wandb-api-key>'
from transformers import AutoModelForCausalLM, AutoTokenizer
import datasets
from tqdm.auto import tqdm
import numpy as np
import torch
# from transformers import AdamW
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss,MSELoss, NLLLoss, KLDivLoss
import json
import random
import matplotlib.pyplot as plt
import transformers
import sys, os
sys.path.append('../.')
sys.path.append('.')
from utils.lora import LoRANetwork
from utils.metrics import get_wmdp_accuracy, get_mmlu_accuracy, get_truthfulqa, get_hp_accuracy
import argparse
import lm_eval
from lm_eval import evaluator
from lm_eval.models.huggingface import HFLM
transformers.utils.logging.set_verbosity(transformers.logging.CRITICAL)
import wandb
from peft import PeftModel, PeftConfig

from huggingface_

In [9]:
# Read the prepare_consistency_data.py script
prepare_script_path = os.path.join(repo_path, 'trainscripts', 'prepare_consistency_data.py')
with open(prepare_script_path, 'r') as f:
    prepare_content = f.read()
print("=" * 80)
print("PREPARE_CONSISTENCY_DATA.PY CONTENT")
print("=" * 80)
print(prepare_content)

PREPARE_CONSISTENCY_DATA.PY CONTENT
from transformers import AutoModelForCausalLM, AutoTokenizer
import datasets
from tqdm.auto import tqdm
import numpy as np
import torch
# from transformers import AdamW
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss,MSELoss, NLLLoss, KLDivLoss
import json
import random
import matplotlib.pyplot as plt
import transformers
import sys, os
sys.path.append('../.')
sys.path.append('../../.')
sys.path.append('.')
from utils.lora import LoRANetwork
from utils.metrics import get_wmdp_accuracy, get_mmlu_accuracy, get_truthfulqa
import argparse
import lm_eval
from lm_eval import evaluator
from lm_eval.models.huggingface import HFLM
transformers.utils.logging.set_verbosity(transformers.logging.CRITICAL)
from transformers import (AutoModelForCausalLM, AutoTokenizer)
import numpy as np
import torch
import argparse
from transformers import (LogitsProcessor, LogitsProcessorList, TemperatureLogitsWarper, TopPLogitsWarper)
import torch.nn.functiona

In [10]:
# Read the metrics.py file
metrics_path = os.path.join(repo_path, 'utils', 'metrics.py')
with open(metrics_path, 'r') as f:
    metrics_content = f.read()
print("=" * 80)
print("METRICS.PY CONTENT")
print("=" * 80)
print(metrics_content)

METRICS.PY CONTENT
## USE LM-EVAL-HARNESS 
# THIS IS NOT FOR MAIN EVALUATION

from transformers import AutoModelForCausalLM, AutoTokenizer
import datasets
from tqdm.auto import tqdm
import numpy as np
import torch
import sys
import os
import torch as t
import csv
import json
import random

ans_map = {
    'A': 0,
    'B': 1,
    'C': 2,
    'D': 3
}

def prepare_data(data, batch_size=8):
    """
    Return a generator of batches of the form (text_batch, answers_batch)
    """
    batch = []
    for row in data:

        question = f"""\
The following are multiple choice questions (with answers).

{row[0]}
A. {row[1]}
B. {row[2]}
C. {row[3]}
D. {row[4]}
Answer:
"""
        ans = row[5]
        batch.append((question, ans_map[ans]))
        if len(batch) == batch_size:
            yield batch
            batch = []


def prepare_data_wmdp(data, batch_size=8):
    """
    Return a generator of batches of the form (text_batch, answers_batch)
    """
    batch = []
    for row in data:
    

In [11]:
# Read the lora.py file
lora_path = os.path.join(repo_path, 'utils', 'lora.py')
with open(lora_path, 'r') as f:
    lora_content = f.read()
print("=" * 80)
print("LORA.PY CONTENT")
print("=" * 80)
print(lora_content)

LORA.PY CONTENT
# ref:
# - https://github.com/cloneofsimo/lora/blob/master/lora_diffusion/lora.py
# - https://github.com/kohya-ss/sd-scripts/blob/main/networks/lora.py

import os
import math
from typing import Optional, List, Type, Set, Literal

import torch
import torch.nn as nn
from safetensors.torch import save_file


LORA_PREFIX = "lora"



TRAINING_METHODS = Literal[
    "attn",  # train all attn layers
    "mlp",  # train all mlp layers
    "full",  # train all layers
]


class LoRAModule(nn.Module):
    """
    replaces forward method of the original Linear, instead of replacing the original Linear module.
    """

    def __init__(
        self,
        lora_name,
        org_module: nn.Module,
        multiplier=1.0,
        lora_dim=1,
        alpha=1,
    ):
        """if alpha == 0 or None, alpha is rank (no scaling)."""
        super().__init__()
        self.lora_name = lora_name
        self.lora_dim = lora_dim

        if "Linear" in org_module.__class__.__name__:
     

## Analysis of Plan vs Implementation

### Plan Summary (from plan.md)

**Objective:** Develop a principled approach for erasing broad conceptual knowledge from language models.

**Hypotheses:**
1. Language models can act as their own critics using self-classification
2. Effective concept erasure requires modifying the model to reduce likelihood of generating concept-containing text
3. Low-rank adapters applied to early model layers enable precise knowledge modification

**Methodology:**
1. ELM uses introspective classification with two context prompts: c− (expert) and c+ (novice)
2. Three loss terms: Lerase, Lretain, and optionally Lfluency
3. LoRA trained on early model layers (layers 4-7 for Zephyr-7B, rank 4, η=500)
4. Training data: erase datasets and retain datasets with expert/novice context prompts

**Experiments:**
1. WMDP biosecurity and cybersecurity concept erasure
2. Ablation study of loss components
3. Robustness to adversarial attacks
4. Internal representation analysis
5. Harry Potter literary domain erasure
6. Hyperparameter analysis

In [12]:
# Now let's carefully analyze the plan and implementation to check CS1 and CS2

print("=" * 80)
print("CS1: CONCLUSION VS ORIGINAL RESULTS ANALYSIS")
print("=" * 80)

print("""
The plan.md file contains the following main results from experiments:

1. WMDP Experiment Results (from plan):
   - ELM achieves near-random performance on WMDP (Bio: 29.7-33.7%, Cyber: 26.6-28.2%)
   - MMLU maintained (75.2-78.8%)
   - MT-Bench (7.1-7.9)
   - R-PPL (4.3-10.9) better than baselines RMU and RepNoise

2. Ablation Study Results (from plan):
   - Lerase is crucial for erasure (w/o: 64.8% Bio vs. 29.7% with)
   - Lretain vital for specificity (w/o: 23.6% MMLU vs. 56.6% with)
   - Lfluency essential for coherence (w/o: 29.8 R-PPL vs. 11.0 with)

3. Harry Potter Results (from plan):
   - ELM achieves 38.3% HP-MCQ (better than WHP 58.6% and RMU 51.0%)
   - MMLU: 45.3%
   - R-PPL: 3.4

4. Hyperparameter Analysis (from plan):
   - Early layers (4-7) more effective than late layers
   - Optimal config: rank 4, η=500, layers 4-7

CHECKING THE IMPLEMENTATION NOTEBOOK (inference.ipynb):
- The notebook demonstrates model loading and inference
- It shows how to load a pre-trained PEFT (LoRA) model
- It provides generation capabilities
- NO ACTUAL RESULTS ARE RECORDED in the notebook
- The notebook is a demonstration/testing notebook, not an evaluation notebook

VERDICT FOR CS1:
The inference.ipynb notebook does NOT contain any recorded experimental results.
It is purely a demonstration notebook showing how to use the trained models.
The conclusions in plan.md reference specific numerical results that are NOT present
in any notebook in the repository.

There is no contradiction because there are no recorded results in the notebook to compare.
However, we cannot verify the conclusions because the actual results are not present.

Since CS1 evaluates whether "conclusions match the results originally recorded in the 
code implementation notebook", and the notebook contains NO recorded results, 
the conclusions CANNOT be verified from this repository.

This is a FAIL for CS1 because we cannot verify the conclusions from the repository.
""")

print("\n" + "=" * 80)
print("CS2: IMPLEMENTATION FOLLOWS THE PLAN ANALYSIS")
print("=" * 80)

print("""
Checking if all plan steps appear in the implementation:

METHODOLOGY FROM PLAN:
1. ✅ ELM uses introspective classification with two context prompts (c− and c+)
   - FOUND in erase.py: positive_concept_prompt (expert) and negative_concept_prompt (novice)
   - Function get_edit_vector() implements this

2. ✅ Three loss terms: Lerase, Lretain, and Lfluency
   - FOUND in erase.py:
     * erase_loss_scale controls Lerase
     * retain_loss_scale controls Lretain  
     * consistence_loss_scale controls Lfluency (consistency/fluency loss)

3. ✅ LoRA trained on early model layers (layers 4-7 for Zephyr-7B, rank 4, η=500)
   - FOUND in erase.py:
     * layers_to_train parameter (default '4,8')
     * lora_rank parameter (default 256, but configurable)
     * eta parameter (default 1000)
   
4. ✅ Training data: erase datasets and retain datasets with expert/novice context prompts
   - FOUND in erase.py: prepare_prompts() function handles multiple datasets
     * Dataset index 0: WMDP-Bio
     * Dataset index 1: WMDP-Cyber
     * Dataset index 2: Harry Potter

EXPERIMENTS FROM PLAN:
1. ✅ WMDP biosecurity and cybersecurity concept erasure
   - FOUND: dataset_idx supports '0' (bio) and '1' (cyber)
   - Evaluation with lm_eval for wmdp_bio and wmdp_cyber

2. ✅ Ablation study of loss components
   - FOUND: erase_loss_scale, retain_loss_scale, consistence_loss_scale 
     can be set to 0 to ablate each component

3. ❌ Robustness to adversarial attacks (GCG, BEAST)
   - NOT FOUND: No GCG or BEAST attack code in the repository

4. ❌ Internal representation analysis (probing accuracy, activation norms)
   - NOT FOUND: No probing or activation analysis code

5. ✅ Harry Potter literary domain erasure
   - FOUND: dataset_idx '2' supports Harry Potter dataset
   - get_hp_accuracy() function in metrics.py

6. ✅ Hyperparameter analysis (LoRA rank, η, layer range)
   - FOUND: All hyperparameters are configurable:
     * --lora_rank
     * --eta
     * --layers_to_train

VERDICT FOR CS2:
Two experiments mentioned in the plan are NOT implemented in the code:
1. Robustness to adversarial attacks (GCG, BEAST)
2. Internal representation analysis (probing accuracy, activation norms)

These are non-trivial experiments that are documented in the plan but have no
corresponding implementation in the repository.

This is a FAIL for CS2 because not all plan steps appear in the implementation.
""")


CS1: CONCLUSION VS ORIGINAL RESULTS ANALYSIS

The plan.md file contains the following main results from experiments:

1. WMDP Experiment Results (from plan):
   - ELM achieves near-random performance on WMDP (Bio: 29.7-33.7%, Cyber: 26.6-28.2%)
   - MMLU maintained (75.2-78.8%)
   - MT-Bench (7.1-7.9)
   - R-PPL (4.3-10.9) better than baselines RMU and RepNoise

2. Ablation Study Results (from plan):
   - Lerase is crucial for erasure (w/o: 64.8% Bio vs. 29.7% with)
   - Lretain vital for specificity (w/o: 23.6% MMLU vs. 56.6% with)
   - Lfluency essential for coherence (w/o: 29.8 R-PPL vs. 11.0 with)

3. Harry Potter Results (from plan):
   - ELM achieves 38.3% HP-MCQ (better than WHP 58.6% and RMU 51.0%)
   - MMLU: 45.3%
   - R-PPL: 3.4

4. Hyperparameter Analysis (from plan):
   - Early layers (4-7) more effective than late layers
   - Optimal config: rank 4, η=500, layers 4-7

CHECKING THE IMPLEMENTATION NOTEBOOK (inference.ipynb):
- The notebook demonstrates model loading and infe

## Evaluation Results

### CS1: Conclusion vs Original Results - **FAIL**

**Reason:** The inference.ipynb notebook does NOT contain any recorded experimental results. It is purely a demonstration notebook showing how to load and use the trained models. The conclusions documented in plan.md contain specific numerical results (WMDP-Bio: 29.7-33.7%, MMLU: 75.2-78.8%, etc.) that are NOT recorded in any notebook in the repository.

Since CS1 evaluates whether "conclusions match the results originally recorded in the code implementation notebook", and the notebook contains NO recorded results, the conclusions CANNOT be verified from this repository.

### CS2: Implementation Follows the Plan - **FAIL**

**Reason:** Two experiments mentioned in the plan are NOT implemented in the code:

1. **Robustness to adversarial attacks (GCG, BEAST)** - The plan mentions testing ELM against GCG (5000 iterations) and BEAST attacks, but no corresponding implementation exists in the repository.

2. **Internal representation analysis** - The plan mentions analyzing probing accuracy and activation norms across layers, comparing ELM vs RMU vs RepNoise. No probing or activation analysis code exists in the repository.

### Detailed Mapping

| Plan Element | Implementation Status | Location |
|--------------|----------------------|----------|
| ELM introspective classification (c−/c+) | ✅ Implemented | `erase.py:get_edit_vector()` |
| Three loss terms (Lerase, Lretain, Lfluency) | ✅ Implemented | `erase.py:train_elm()` |
| LoRA on early layers | ✅ Implemented | `erase.py:layers_to_train` |
| WMDP Bio/Cyber erasure | ✅ Implemented | `erase.py:dataset_idx` |
| Ablation study | ✅ Configurable | `erase.py:loss_scale params` |
| Adversarial attacks (GCG/BEAST) | ❌ Missing | - |
| Internal representation analysis | ❌ Missing | - |
| Harry Potter erasure | ✅ Implemented | `erase.py:dataset_idx='2'` |
| Hyperparameter analysis | ✅ Configurable | `erase.py:args` |

## Summary

### Binary Checklist Results

| Checklist Item | Result | Rationale |
|----------------|--------|-----------|
| **CS1: Conclusion vs Original Results** | **FAIL** | The notebook (inference.ipynb) contains no recorded experimental results. The conclusions in plan.md contain specific numerical values that cannot be verified from any notebook in the repository. |
| **CS2: Implementation Follows Plan** | **FAIL** | Two plan experiments are missing from the implementation: (1) Robustness to adversarial attacks (GCG, BEAST), (2) Internal representation analysis (probing accuracy, activation norms). |

### Missing Elements Summary

**For CS1 (Conclusions cannot be verified):**
- No experimental results recorded in notebooks
- Plan contains specific metrics (WMDP-Bio: 29.7-33.7%, MMLU: 75.2-78.8%, etc.) that are not present in any notebook

**For CS2 (Missing implementations):**
1. GCG adversarial attack testing (mentioned: 5000 iterations)
2. BEAST adversarial attack testing
3. Linear probe accuracy analysis across layers
4. Activation norm distribution analysis

In [13]:
# Create the evaluation directory
eval_dir = os.path.join(repo_path, 'evaluation')
os.makedirs(eval_dir, exist_ok=True)
print(f"Created evaluation directory: {eval_dir}")

Created evaluation directory: /net/scratch2/smallyan/erasing-llm_eval/evaluation


In [14]:
# Create the consistency_evaluation.json file
import json

evaluation_results = {
    "Checklist": {
        "CS1_Results_vs_Conclusion": "FAIL",
        "CS2_Plan_vs_Implementation": "FAIL"
    },
    "Rationale": {
        "CS1_Results_vs_Conclusion": "The inference.ipynb notebook contains NO recorded experimental results. It is purely a demonstration notebook showing how to load and use the trained models. The conclusions documented in plan.md contain specific numerical results (WMDP-Bio: 29.7-33.7%, MMLU: 75.2-78.8%, MT-Bench: 7.1-7.9, R-PPL: 4.3-10.9, etc.) that are NOT recorded in any notebook in the repository. Since CS1 evaluates whether conclusions match the results originally recorded in the code implementation notebook, and the notebook contains NO recorded results, the conclusions CANNOT be verified from this repository.",
        "CS2_Plan_vs_Implementation": "Two experiments mentioned in the plan are NOT implemented in the code: (1) Robustness to adversarial attacks - The plan mentions testing ELM against GCG (5000 iterations) and BEAST attacks, but no corresponding implementation exists in the repository. (2) Internal representation analysis - The plan mentions analyzing probing accuracy and activation norms across layers comparing ELM vs RMU vs RepNoise, but no probing or activation analysis code exists in the repository. These are non-trivial experiments that are documented in the plan but have no corresponding implementation."
    }
}

# Write to file
json_path = os.path.join(eval_dir, 'consistency_evaluation.json')
with open(json_path, 'w') as f:
    json.dump(evaluation_results, f, indent=4)

print(f"Created: {json_path}")
print("\nContent:")
print(json.dumps(evaluation_results, indent=4))

Created: /net/scratch2/smallyan/erasing-llm_eval/evaluation/consistency_evaluation.json

Content:
{
    "Checklist": {
        "CS1_Results_vs_Conclusion": "FAIL",
        "CS2_Plan_vs_Implementation": "FAIL"
    },
    "Rationale": {
        "CS1_Results_vs_Conclusion": "The inference.ipynb notebook contains NO recorded experimental results. It is purely a demonstration notebook showing how to load and use the trained models. The conclusions documented in plan.md contain specific numerical results (WMDP-Bio: 29.7-33.7%, MMLU: 75.2-78.8%, MT-Bench: 7.1-7.9, R-PPL: 4.3-10.9, etc.) that are NOT recorded in any notebook in the repository. Since CS1 evaluates whether conclusions match the results originally recorded in the code implementation notebook, and the notebook contains NO recorded results, the conclusions CANNOT be verified from this repository.",
        "CS2_Plan_vs_Implementation": "Two experiments mentioned in the plan are NOT implemented in the code: (1) Robustness to adversa

In [15]:
# Now let's copy the current notebook to the evaluation directory as self_matching.ipynb
import shutil

# Get the current notebook path
current_notebook = '/home/smallyan/eval_agent/notebooks/2026-01-12-11-09_ConsistencyEvaluation.ipynb'
target_notebook = os.path.join(eval_dir, 'self_matching.ipynb')

# Copy the notebook
shutil.copy(current_notebook, target_notebook)
print(f"Copied notebook to: {target_notebook}")

Copied notebook to: /net/scratch2/smallyan/erasing-llm_eval/evaluation/self_matching.ipynb


In [16]:
# Verify both files exist
print("Files created in evaluation directory:")
for f in os.listdir(eval_dir):
    filepath = os.path.join(eval_dir, f)
    print(f"  {f} ({os.path.getsize(filepath)} bytes)")

Files created in evaluation directory:
  consistency_evaluation.json (1396 bytes)
  self_matching.ipynb (155192 bytes)


In [17]:
# Final summary
print("=" * 80)
print("CONSISTENCY EVALUATION COMPLETE")
print("=" * 80)
print(f"""
Repository: /net/scratch2/smallyan/erasing-llm_eval

Files Created:
1. {eval_dir}/self_matching.ipynb
2. {eval_dir}/consistency_evaluation.json

RESULTS:
- CS1 (Conclusion vs Original Results): FAIL
  The notebook contains no recorded experimental results to verify against the conclusions.

- CS2 (Implementation Follows Plan): FAIL
  Two plan experiments are missing from the implementation:
  1. Robustness to adversarial attacks (GCG, BEAST)
  2. Internal representation analysis (probing accuracy, activation norms)
""")

CONSISTENCY EVALUATION COMPLETE

Repository: /net/scratch2/smallyan/erasing-llm_eval

Files Created:
1. /net/scratch2/smallyan/erasing-llm_eval/evaluation/self_matching.ipynb
2. /net/scratch2/smallyan/erasing-llm_eval/evaluation/consistency_evaluation.json

RESULTS:
- CS1 (Conclusion vs Original Results): FAIL
  The notebook contains no recorded experimental results to verify against the conclusions.

- CS2 (Implementation Follows Plan): FAIL
  Two plan experiments are missing from the implementation:
  1. Robustness to adversarial attacks (GCG, BEAST)
  2. Internal representation analysis (probing accuracy, activation norms)

